In [7]:
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
import os

db_path = os.path.abspath("resources/Chinook.db")
print(db_path)

E:\code\langchain-basics\resources\Chinook.db


In [2]:
from langchain.tools import tool
from typing import Dict, Any
from sqlalchemy import exc
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:
    """Obtain information from the database using SQL queries"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [3]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest,
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Dynamically called tools based on the runtime context"""
    user_role = request.runtime.context.user_role

    if user_role == "internal":
        pass # internal users get access t all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools)
    
    return handler(request)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search,sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are there in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have access to your database yet. Could you tell me which database you’re referring to and how you want the count done?

Helpful details:
- Database type (e.g., MySQL, PostgreSQL, SQLite, MongoDB, etc.)
- The table/collection name (e.g., artists) and any filters (e.g., only active artists)
- Whether you want an exact count or a distinct count of IDs

Common options:
- SQL (e.g., MySQL/PostgreSQL/SQLite): SELECT COUNT(*) AS artist_count FROM artists;
- PostgreSQL distinct: SELECT COUNT(DISTINCT id) FROM artists;
- MongoDB: db.artists.countDocuments({});

If you can share the schema or give me a way to run a query (connection details or an API endpoint), I can provide the exact command or script you need.


In [7]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are there in the database?")]},
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

There are 275 artists in the database.


In [8]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Who won the IPL in 2024?")]},
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

Kolkata Knight Riders (KKR) won the 2024 IPL (the Tata IPL 2024), defeating Sunrisers Hyderabad in the final.


In [9]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Who won the Bundesliga in 2024?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

Bayer Leverkusen won the Bundesliga for the 2023–24 season, which was decided in 2024. It was Leverkusen’s first Bundesliga title. If you meant the 2024–25 season, that title went to Bayern Munich (decided in 2025).
